# CrewAI Batasan Iterasi

In [1]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

In [2]:
# Setup LLM
llm_crewai = LLM(
    model="ollama/qwen3:0.6b-q4_K_M",
    base_url="http://localhost:11434",
    temperature=0.7
)

# Define tools
class SearchToolInput(BaseModel):
    """Input for SearchTool."""
    query: str = Field(..., description="Search query")

class SearchTool(BaseTool):
    name: str = "Search Tool"
    description: str = "Search for information"
    args_schema: Type[BaseModel] = SearchToolInput

    def _run(self, query: str) -> str:
        return f"Search results for: {query}. AI Agents are autonomous systems..."

class CalculatorToolInput(BaseModel):
    """Input for CalculatorTool."""
    expression: str = Field(..., description="Mathematical expression to calculate")

class CalculatorTool(BaseTool):
    name: str = "Calculator Tool"
    description: str = "Calculate mathematical expression"
    args_schema: Type[BaseModel] = CalculatorToolInput

    def _run(self, expression: str) -> str:
        try:
            result = eval(expression)
            return f"Result: {result}"
        except:
            return "Invalid expression"

search_tool = SearchTool()
calculator_tool = CalculatorTool()

# Define Agents
researcher = Agent(
    role='Researcher',
    goal='Search for comprehensive information about the given topic.',
    backstory="You are a meticulous researcher. You gather raw data.",
    tools=[search_tool],
    llm=llm_crewai,
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

analyzer = Agent(
    role='Analyzer',
    goal='Analyze the findings provided by the researcher and draw insights.',
    backstory="You are an analytical thinker. You don't search, you interpret data.",
    tools=[calculator_tool],
    llm=llm_crewai,
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

# Define Tasks
task_research = Task(
    description="Research about AI agents and their applications.",
    expected_output="A summary of key findings about AI agents.",
    agent=researcher
)

task_analysis = Task(
    description="Analyze the research findings. Provide conclusion and potential future trends.",
    expected_output="A final analytical report with conclusions.",
    agent=analyzer,
    context=[task_research]
)

# Build CrewAI Workflow
crew = Crew(
    agents=[researcher, analyzer],
    tasks=[task_research, task_analysis],
    process=Process.sequential,
    verbose=True
)

In [1]:
initial_input = {
    "topic": "Research about AI agents and their applications"
}

# Execute
result = crew.kickoff()

print("CrewAI Result:")
print(f"Total agents: {len(crew.agents)}")
print(f"Final result: {result}")


NameError: name 'crew' is not defined